# 07 — Decision Framework

**Owner:** Person B (decision track).

**Rubric line:** Decision rule + threshold linked to a clear business
action + pilot plan.

Translates calibrated probabilities into an executable contact policy.
Outputs the headline decision rule that the memo and slide deck quote.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 7.1 — Load test predictions

In [ ]:
import joblib
train_df, test_df = data.load_interim()
y_test = test_df[config.TARGET_COL].values
lgbm_artifact = joblib.load(config.MODELS_DIR / 'improved_lgbm.joblib')
y_proba = lgbm_artifact['proba_test']


## 7.2 — Cost / value assumptions

Documented in `src/config.py`. Default: €5 per contact, €120 per
successful subscription. These are NOT facts — notebook 08 stress-tests
them. If you change them, log the rationale in `ai_usage_log.md`.


In [ ]:
print(f'Cost per contact:        €{config.COST_PER_CONTACT_EUR:.2f}')
print(f'Value per conversion:    €{config.VALUE_PER_CONVERSION_EUR:.2f}')
print(f'Contact budget fraction: {config.CONTACT_BUDGET_FRACTION:.0%}')


## 7.3 — Expected value vs. threshold

In [ ]:
ev_df = metrics.ev_curve(y_test, y_proba)
ev_df.to_csv(config.TABLES_DIR / 'ev_curve.csv', index=False)


## 7.4 — Build the decision rule (top-K policy)

In [ ]:
rule = decision.build_decision_rule(y_test, y_proba)
print(rule.describe())


## 7.5 — Compare against the unconstrained EV-max threshold

In [ ]:
ev_max_t = decision.threshold_max_ev(y_test, y_proba)
ev_max = metrics.expected_value(y_test, y_proba, ev_max_t)
print(f'Unconstrained EV-max threshold: {ev_max_t:.3f}')
print(ev_max)


## 7.6 — Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
viz.plot_ev_curve(ev_df, ax=axes[0], mark_threshold=rule.threshold)
lift = metrics.decile_lift(y_test, y_proba)
viz.plot_decile_lift(lift, ax=axes[1])
viz.save_fig(fig, '07_decision_framework')


## 7.7 — Pilot plan (memo-ready)

Sketch a 90-day pilot:

- **Scope:** which region / segment / cycle.
- **Treatment vs. control:** randomise contact within the top decile to
  measure incremental lift over current practice.
- **KPI:** subscription rate uplift in the targeted decile vs. control.
- **Timeline:** 30 days build / 60 days run / 30 days analyse.
- **Decision criteria for full rollout.**
